# Does teacher training still work on early fine-tunes, and applied again along the fine-tune?

*E41, `experiments/teacher_training/path.py`, 2026-10-28. Record:
`reports/analyses/2026-10-28_analysis_e41-teacher-path.md`. Plan: `reports/plans/2026-10-28_plan_e41-teacher-path.md`.
Follows E22 and E34. Independent check: `reports/reviews/2026-10-28_review_e41-check.md`, corrections applied.*

## The answer

Partly. We copied one fine-tune, stopped at three points along its run, into the subcomponents with Δ fixed. Every copy
kept the behaviour. The adversarial check can tell a copy from doing nothing only at the earliest point, where doing
nothing fails. There one copy passed by about 1% and the emoticon copy was inconclusive, between the lines. A chain of
three copies stayed within the filed tolerances of single copies. Its weight change was larger and its adversarial
ratio slightly higher.

*Amendment 1, at the end: at two more batch orders the step-10 emoticon copy passed once and was inconclusive twice. It sits at the pass line; its reading varied more across orders than finished copies' do, with one adversary start per reading.*

## What we were trying to test

The decomposition writes each weight matrix as its alive subcomponents plus a residual Δ, and Γ decides which
subcomponents matter on each input. **Teacher training** copies a fine-tune into it. It trains only the subcomponents' read
and write vectors. The aim is that the model built from them imitates the fine-tuned model, the teacher. Γ and Δ stay
frozen, so Δ cannot absorb the update. In E34 this kept the behaviour and passed the adversarial check on six
fine-tunes. Every one of them had run to its end, 1,000 steps.

`GOALS.md` asks about small updates, and about a map applied again and again along training. So we took E34's Adam
fine-tune and stopped it early. One run at a constant rate gives three teachers: the model after 10 steps, after 100
and after 1,000. We used two tasks. On the emoticon task an emoticon is followed by `o`. On the second task, " which"
is followed by " was".

Each teacher was copied two ways, each copy trained for 1,000 steps:

```
fine-tune        unchanged ──10 steps──▶ T10 ──90──▶ T100 ──900──▶ T1000

one-shot copies  unchanged ⇒ copy of T10     unchanged ⇒ copy of T100     unchanged ⇒ copy of T1000

chained copies                copy of T10 ⇒ chained copy of T100 ⇒ chained copy of T1000
```

- **One-shot copy.** Start from the unchanged model's subcomponents and train against one teacher. The step-1,000
  one-shot copy is E34's own, rerun as a check that nothing in the code path changed.
- **Chained copy.** Start from the step-10 copy and train against the step-100 teacher, then start from that copy and
  train against the step-1,000 teacher. Each link uses the same recipe and the same batches, with a fresh optimiser.
  If the start carries error forward, the chained copy ends worse than the one-shot copy of the same teacher.

Each copy learns from one of two kinds of text. **Task text** is the fine-tune's own training windows. **Pile text** is
ordinary text, in which the emoticon trigger is very rare; we used it on the emoticon task only.

Every state is read at its last step on four numbers:

- **Adversarial ratio.** An adversary spends 20 steps searching for the most damaging mask over the subcomponents.
  It reads 256 rows of Pile text. The ratio is the damage it finds at the state over the damage at the unchanged
  model, so 1 means no change. On this card it passes at or below 1.40 and fails at or above 1.61. Between the two it
  is inconclusive.
- **Held-out P(target).** The probability of the target token at the trigger, on windows no copy trained on. The
  unchanged model gives 0.004 on emoticon and 0.04 on the second task.
- **Gap share.** How far the state's predictions still are from the teacher's on held-out task windows. It is a share
  of the unchanged model's distance: the unchanged model reads 1 and a perfect copy reads 0.
- **Δ's share.** The share of the state's weight change that sits in Δ. Frozen, it must stay near zero.

The reference for every state is **do-nothing**: the teacher's whole update put into Δ, with the subcomponents and Γ
untouched. It keeps the behaviour exactly, but Δ carries everything, so it cannot meet the weak version. Its
adversarial ratio says whether the check can see the update at all. E22 found the step-10 emoticon update left in Δ
fails the check at 2.52. That is this run's positive control: the check can fail a state of this kind.

Controls ran first, and all passed. The unchanged model read its known noise floor. The regenerated step-1,000
teachers and their one-shot copies matched E34's to four digits. A chained start reproduced its starting copy exactly
before any link trained.

## What we expected, and why

- **Copies of early teachers from task text keep the behaviour and pass**, on both tasks. All six of E34's task-text
  copies passed, and the early teachers are smaller. We leaned high. The risk we named was the step-10 teacher on the
  " which" task, which already does more damage elsewhere than any other teacher here.
- **Copies of early teachers from Pile text learn the behaviour.** A low-confidence lean. E34's Pile-text copy of the
  finished emoticon fine-tune reached 0.90, right at the line.
- **The chain is no worse than the one-shot copy**, on task text and on Pile text. Each link is a full E34 copy and only
  its start differs. We leaned high on task text, and medium on Pile text, where E34 found copies land at different
  weight changes from batch order alone.

## Does a copy of an early fine-tune pass where doing nothing fails?

![adversarial ratio against the fine-tune's step](../experiments/teacher_training/figures/e41_faithful.png)

**Reading it.** On the " which" task it passes, but only by 1.4%; on emoticon it is inconclusive. The grey squares are
do-nothing, and at step 10 they fail on both tasks, at 2.52 and 2.16. So the check can see these early updates. The
step-10 copy from task text reads 1.38 on the " which" task, just under the pass line and well inside the noise. On
emoticon it reads 1.55, between the lines. At step 100 every copy passes, but so does do-nothing, so those cells cannot
tell a real map from the trivial one. At step 1,000 do-nothing is only inconclusive. So step 10 is the one step where
the check separates a map from doing nothing. The Pile-text copy of the same step-10 emoticon fine-tune passes at 1.13,
so the miss belongs to the task-text copy, not to the teacher alone.

### The one miss, read two ways

We filed that every early copy from task text would pass, and the step-10 emoticon copy did not. It failed in the one
cell where the check had full power. Two readings hold at once, and we do not re-score either.

- **Against the pass line**, the map falls short on the smallest and most fragile update we gave it, on one of two
  tasks. The other step-10 copy passes by a margin of only 0.02.
- **Against doing nothing**, the copy removes about two-thirds of the extra damage the adversary finds when the same
  update sits in Δ. Its behaviour is intact. It puts 0.9994 on the target and is closer to its teacher than any later
  copy.

The copy reads 11% above the pass line and only 4% below the fail line. That 4% is inside the unchanged model's own
spread across adversary starts. Its extra damage also sits in one of the two batches of rows the adversary reads, where
it would read fail on its own. So a rerun could land in pass, inconclusive or fail. To pass it would need a drop about
as large as the largest spread a second batch order caused in E34.

The closed-form adjoint split did better on this same step-10 update. In E22 it passed at 1.06 with the target's
probability at 0.93 and Δ fixed. It loses the behaviour later in the fine-tune, falling to 0.10 by step 1,000, where
teacher training keeps it.

## Do copies of early fine-tunes keep the behaviour?

![held-out P(target) and gap share against the fine-tune's step](../experiments/teacher_training/figures/e41_behaviour.png)

**Reading it.** Yes, every copy. In the left panel every copy puts 0.90 or more on the target, where the unchanged model
puts 0.004 or 0.04. The black stars and crosses are the fine-tunes themselves, which do-nothing reproduces exactly. In
the right panel every copy closes its teacher gap to under 3% of the unchanged model's. Every copy of an early teacher
gets it under half a percent. The Pile-text copies did better on early teachers than on the finished one. Copies of
the step-10 and step-100 fine-tunes reach 0.998 and 0.995. The copy of the finished one reaches 0.90. They also learn
faster. A tenth of the way through training, the Pile-text copy of the step-10 teacher already holds 0.99. The copy of
the finished one holds 0.34.

## Does a chain of copies along the fine-tune pile up error?

![chained against one-shot copies: cosine and size of the weight change](../experiments/teacher_training/figures/e41_chain.png)

**Reading it.** Not on any number we scored, over a chain of three checkpoints with 1,000 training steps per copy. In
the first two figures the hollow diamonds, the chained copies, sit beside the one-shot copies. Every chained copy
passes, keeps the behaviour and leaves Δ alone, within the filed tolerances. But on task text the chained ratios run 4%
to 5% above the one-shot copies', in all four cells. Each is inside the noise, yet the sign is the same every time. At
step 1,000 the chain is also a little closer to its teacher than the one-shot copy.

This figure shows the chain does not land where the one-shot copy does. In the left panel, the two step-1,000 weight
changes have a cosine of 0.65 to 0.86. The dashed lines are two copies of another teacher at different batch orders,
from E34. In the right panel the chained change is 20% to 36% larger than the one-shot copy's, though still smaller
than the fine-tune's own. Relative to the one-shot copy, its size and its damage elsewhere grew from the second link to
the third in every group. That is the pattern accumulation would show. Two links cannot tell a trend from noise, and
we filed nothing on it, so the next run must.

In [1]:
# Every copy at its last step, from the results file (no model).
import json
from pathlib import Path

r = json.loads(Path("../experiments/teacher_training/results/teacher_training_e41.json").read_text())
name = {"emoticon": "emoticon", "r1_t534_v369": "which/was"}
print(f"{'copy':38s} {'P(target)':>9s} {'ratio':>6s} {'verdict':>13s} {'gap share':>9s}")
for t, task in r["tasks"].items():
    for arm, text in (("A", "task"), ("B", "Pile")):
        for kind in ("oneshot", "chain"):
            for k, s in task.get(arm, {}).get(kind, {}).items():
                label = f"{name[t]}, {text} text, T{k}, {'chained' if kind == 'chain' else 'one-shot'}"
                print(f"{label:38s} {s['p_target']:9.4f} {s['pgd']:6.3f} {s['verdict']:>13s} {s['gap_share']:9.4f}")

copy                                   P(target)  ratio       verdict gap share
emoticon, task text, T10, one-shot        0.9994  1.547  inconclusive    0.0013
emoticon, task text, T100, one-shot       0.9989  1.016          pass    0.0014
emoticon, task text, T1000, one-shot      0.9968  0.959          pass    0.0056
emoticon, task text, T100, chained        0.9982  1.054          pass    0.0019
emoticon, task text, T1000, chained       0.9979  1.000          pass    0.0033
emoticon, Pile text, T10, one-shot        0.9984  1.134          pass    0.0038
emoticon, Pile text, T100, one-shot       0.9954  0.954          pass    0.0049
emoticon, Pile text, T1000, one-shot      0.9021  1.049          pass    0.0266
emoticon, Pile text, T100, chained        0.9965  0.967          pass    0.0043
emoticon, Pile text, T1000, chained       0.9030  1.027          pass    0.0251
which/was, task text, T10, one-shot       0.9892  1.376          pass    0.0010
which/was, task text, T100, one-shot    

## What we learned

Teacher training keeps the behaviour of every early teacher with Δ untouched, from task text on two tasks and from Pile
text on emoticon. On the adversarial check, at the one step where doing nothing fails, one copy passed narrowly and the
other was inconclusive. The step-10 task-text copies read the highest ratios of any trained state. So the small updates
`GOALS.md` asks about are where this map is weakest, on one seed, one batch order and one adversary start. The adjoint
split passed on that same update but loses the behaviour later, so neither map yet covers the whole fine-tune. A chain
of three copies stayed within the filed tolerances, while its size and damage grew from link to link.

## How compelling is the evidence

- **One draw of everything.** One fine-tune seed, one batch order, one adversary start of 20 steps on 256 Pile rows. In
  E34 a second start once read a fine-tune left in Δ six times higher, so every pass is a lower bound.
- **Two trigger tasks, one optimiser.** Both are behaviours the model half has, and every teacher is an Adam fine-tune.
  The Pile-text copies were run on emoticon only.
- **Power.** The check sees the step-10 updates, since do-nothing fails on both. It does not see the step-100 updates,
  since do-nothing passes there; those cells say nothing about the check.
- **Noise.** The unchanged model varies by 7.5% across adversary starts. In E34 a second batch order moved task-text
  copies by up to 11% and Pile-text copies by up to 25%.
- **The chain had more training**, 2,000 or 3,000 steps in total against the one-shot copy's 1,000. Its ratio was still
  higher on task text, while its gap to the teacher at step 1,000 was lower.
- **The cosine references** come from another teacher, E34's SGD fine-tune on the " which" task. They show the spread
  batch order causes, not this teacher's.

## Surprises

- **A filed prediction failed, in the one cell where the check had full power.** We expected every early task-text copy
  to pass, at high confidence. The step-10 emoticon copy read 1.55, inconclusive, 4% under the fail line. A rerun could
  read it either way, and the plan sends it back for one.
- **The adjoint split beat the copy on the check at step 10**, 1.06 against 1.55 on the same update.
- **Early fine-tunes are easier to copy from Pile text than the finished one.** We filed that they would learn, at low
  confidence, since the finished copy sat at 0.90. The step-10 and step-100 copies reached 0.998 and 0.995.
- **Step-10 copies move further than their teachers.** Their largest weight change exceeds the fine-tune's own. They
  also do as much damage elsewhere as the teacher, where copies of the finished fine-tune do a third to a half of it.
- **The chained change grows link by link**, 20% to 36% over the one-shot copy's at step 1,000, and so does its damage
  elsewhere. We filed nothing on it.

## What next

1. **The step-10 emoticon copy at two more batch orders**, E41 Amendment 1, an approved branch at about $1. The plan's
   branch reads "if its verdict fails"; this copy is inconclusive, so the amendment reads "fails" as "does not pass".
   Its predictions are filed first and cover pass, inconclusive, fail and a mixed result. Either way it goes to the
   user as a question about small updates and the weak version.
2. **Short chain links**, the other approved branch, at about $1.5. A chain over seven checkpoints of the emoticon
   fine-tune, from step 1 to step 1,000, with 100 training steps per link, read at every link. It should file a
   prediction on the size and damage growth seen here. If the ratio, size or damage grows link by link, the map needs
   a reset.

## Scorecard

Filed in `experiments/teacher_training/results/predictions_e41.json` before the run, unedited. Scored by the run's
`score` block and checked against the filed rules.

| id | claim | rule | read | verdict |
|---|---|---|---|---|
| C1 | unchanged model at its known floor | floor 0.0748 within 10% | 0.0748 | pass |
| G1, G3 | step-1,000 teachers and copies match E34 | P within 0.005; size within 0.001, ratio within 15% | equal to four digits | pass ×4 |
| C2 | chained start reproduces its start | first KL ≤ 0.005; weights within 1e-3 | 0.0005; 0 | pass |
| P1 | early task-text copies pass, gap ≤ 0.05 | every cell | emoticon step 10: 1.547, inconclusive; other three pass, the step-10 one by 1.4% | **fails as filed**; of the two cells with power, one pass, one inconclusive |
| P2 | early Pile-text copies learn, emoticon | learnt, between or neither | 0.998 and 0.995, both pass | learnt, as leaned |
| P3 | chain not worse, task text | 4 conditions per cell | 4 of 4 cells | holds |
| P4 | chain not worse, Pile text | 4 conditions per cell | 2 of 2 cells | holds |
| — | Δ untouched | Δ's share ≤ 1e-6 | ≤ 2.3e-9 | pass |

## Not run

- The step-10 emoticon copy at a second and third batch order; short chain links.
- A one-shot copy trained 3,000 steps to match the chain's total.
- SGD or LoRA teachers; other tasks; Pile-text copies on the " which" task.
- A second fine-tune seed; more adversary starts or a longer adversary.
- No states were saved; any follow-up regenerates one in about 3 minutes.

## Provenance

RunPod NVIDIA H100 80GB HBM3, 2026-10-28, job 12:15 to 12:58, 43.2 minutes of job, about $3.2 in all. Results
`experiments/teacher_training/results/teacher_training_e41.json`, committed in a450fde. Log
`experiments/teacher_training/results/large/run_e41.log`. Figures from `experiments/teacher_training/plot_e41.py`.

# Amendment 1: is the step-10 emoticon copy a pass, a fail, or at the line?

*E41 Amendment 1, `experiments/teacher_training/path_amend1.py`, 2026-10-28. Record:
`reports/analyses/2026-10-28_analysis_e41-teacher-path.md` → "Amendment 1". Plan: the same plan → "Amendment 1".
Independent check: `reports/reviews/2026-10-28_review_e41-amend1-check.md`, corrections applied.*

## The answer

At the line. We retrained the copy of the step-10 emoticon fine-tune at two more batch orders. One of the three copies
passes the adversarial check, at 1.28. The other two sit between the lines, at 1.55 and 1.51. Every copy keeps the
behaviour, and each removes 64% to 82% of the excess that leaving the update in Δ shows on the check. Across three
orders the reading spans a fifth. With one adversary start per reading, how much of that is batch order is untested.

## What we were trying to test

Above, the step-10 emoticon copy from task text was the one early copy that did not pass. It read 1.55, between the pass
line at 1.40 and the fail line at 1.61. Was that one unlucky draw, or where this copy sits?

So we made the same copy twice more. Same teacher, same recipe, same 1,000 training steps, same single adversary start.
The only change is the batch order: which task windows go into each training batch, and in what order. E41 used order
0; here orders 1 and 2.

Two references sit beside the new copies. Doing nothing, the step-10 update left in Δ, reads 2.52 and fails: that is
what a copy has to beat. And a copy of the finished fine-tune at order 1 must reproduce an earlier run's copy exactly.
It checks that the new code path is the one that measured how much batch order moves finished copies.

One new reading: the check's 256 Pile rows are scored as two blocks of 128, each with its own adversary. A block's ratio
is the adversary's damage there over the unchanged model's on the same block. The check's ratio is their mean.

## What we expected, and why

- **The middle of the three verdicts would be inconclusive.** Batch order had moved emoticon's finished copies by at
  most 3%, and a pass needed a drop of almost 10%. Medium confidence.
- **Both new readings would land within 11% of the first.** That was the widest batch-order gap seen for any finished
  copy from task text. Medium confidence; the named risk was that the first copy's excess sat in one block.

## Does the step-10 copy pass at other batch orders?

![adversarial ratio of the step-10 copy at three batch orders, beside doing nothing and copies of the finished fine-tune](../experiments/teacher_training/figures/e41_amend1_seeds.png)

**Reading it.** At one order of three. The dark blue dots are the three copies of the step-10 fine-tune. Order 1 passes
at 1.28, by just more than the unchanged model's own spread across adversary starts. Orders 0 and 2 sit between the
dotted lines, at 1.55 and 1.51. The grey dot is doing nothing, at 2.52. Each copy removes 64% to 82% of the excess that
doing nothing shows on the check. The two pale dots on the right are copies of the finished fine-tune at orders 0 and
1, which differ by 3%. The step-10 copies span 21%, and order 1 alone sits 17% below order 0, outside the 11% we filed.

The small triangles split each reading into its two blocks of rows. The first block reads about 1.07 at every order,
close to the unchanged model. The second block carries nearly all the excess: 2.00, 1.47 and 1.92. The dashed grey
line is the unchanged model's highest reading on that block across adversary starts, 1.15. So every order leaves a real
excess on the same 128 rows, and its size varies across orders.

The caveat: batch order is one of two noise sources here, not a shown cause. Each reading had one adversary start. The
copies agree on every number the adversary does not touch, such as damage, gap share and the first block. They differ
only in what the adversary finds on the second block. Whether the copy or the adversary's start moved is untested.

In [2]:
# The step-10 copies at three batch orders, from the two results files (no model).
import json
from pathlib import Path

res = Path("../experiments/teacher_training/results")
a1 = json.loads((res / "teacher_training_e41_amend1.json").read_text())
e41 = json.loads((res / "teacher_training_e41.json").read_text())
order0 = e41["tasks"]["emoticon"]["A"]["oneshot"]["10"]
rows = [("doing nothing", a1["teachers"]["10"]["nothing"]), ("copy, order 0", order0)]
rows += [(f"copy, order {s}", a1["copies"][s]) for s in ("1", "2")]
print(f"{'state':16s} {'ratio':>6s} {'verdict':>13s} {'P(target)':>9s} {'gap share':>9s}")
for name, s in rows:
    print(f"{name:16s} {s['pgd']:6.3f} {s['verdict']:>13s} {s['p_target']:9.4f} {s['gap_share'] if 'gap_share' in s else 0:9.4f}")

state             ratio       verdict P(target) gap share
doing nothing     2.516          fail    0.9993    0.0000
copy, order 0     1.547  inconclusive    0.9994    0.0013
copy, order 1     1.280          pass    0.9991    0.0014
copy, order 2     1.505  inconclusive    0.9990    0.0013


## What we learned

On the step-10 update of this fine-tune, teacher training from task text sits at the pass line. Across three batch
orders it passed once and was inconclusive twice, never failing and always keeping the behaviour. Its reading varied
more across orders than any finished copy's, with the adversary's start not separated. So a single reading of an early
copy near the line can land on either side of it. On both tasks the step-10 copies read closest to the pass line of
any copy; emoticon was read at three orders and " which" at one. On the same update the adjoint split passed in E22,
with P(target) 0.93 against the copies' 0.999, and it loses the behaviour later in the fine-tune.

## How compelling is the evidence

- **Three batch orders**: enough to see a spread, not to estimate one. One fine-tune seed, one task, the 67M model.
- **One adversary start per state**, so a pass is a lower bound, and the start is mixed into the batch-order spread.
- **The excess sits in one block of 128 rows.** Other Pile rows could read the copy differently.
- **The references reproduced exactly**: doing nothing block for block, and the finished copy at order 1 to the digit.

## Surprises

- **The spread prediction failed.** We filed that both new readings would land within 11% of the first, at medium
  confidence. Order 1 read 17% lower. The reading varied more across orders than any finished copy's measured, with
  the adversary's start not separated.
- The median verdict came out inconclusive, as leaned.

## What next

The plan sends this outcome to the user as a decision on the step-10 update. The recommended choice is to accept "at
the line" and read early copies at three batch orders from now on. Extra adversary starts would first tell whether
three starts on one copy could do instead. The card, with costs, is at the end of the report's "Amendment 1" section.

## Scorecard (Amendment 1)

Filed in `experiments/teacher_training/results/predictions_e41_amend1.json` before the run, unedited.

| id | claim | rule | read | verdict |
|---|---|---|---|---|
| C1, G1, G2 | unchanged model, teachers and the seeded finished copy reproduce | as filed | equal to the digit | pass |
| P1 | median verdict over three batch orders | median of pass < inconclusive < fail, unless behaviour lost | inconclusive, pass, inconclusive | **inconclusive**, as leaned |
| P2 | both new orders within 11.1% of 1.547 | 1.375 to 1.718 | 1.280 and 1.505 | **fails** |
| — | Δ untouched | Δ's share ≤ 1e-6 | 2.4e-9, 2.2e-9 | pass |

## Not run (Amendment 1)

- More adversary starts or the 160-step adversary on the step-10 copies; a second set of Pile rows.
- Other orders for the step-100 copies, the " which" step-10 copy and the Pile-text step-10 copy.
- A 3,000-step copy, a start from the adjoint split, a second fine-tune seed.
- Which rows of the second block carry the excess.

RunPod NVIDIA H100 80GB HBM3, 2026-10-28, about 11 minutes of job, about $1.1. Results
`experiments/teacher_training/results/teacher_training_e41_amend1.json`; log
`experiments/teacher_training/results/large/run_e41_amend1.log`; figure from
`experiments/teacher_training/plot_e41_amend1.py`, which also hard-codes order 0's block values from E41's log
`experiments/teacher_training/results/large/run_e41.log`.

# Amendment 2: does teacher training, applied every 100 steps along the fine-tune, stay faithful?

*E41 Amendment 2, `experiments/teacher_training/path_amend2.py`, 2026-10-28. Record:
`reports/analyses/2026-10-28_analysis_e41-teacher-path.md` → "Amendment 2". Plan: the same plan → "Amendment 2".*

## The answer

Yes, within what the check can see. We copied the emoticon fine-tune into the subcomponents at seven checkpoints, 100
training steps each, each copy starting from the last. Every copy kept the behaviour and passed the adversarial check.
Leaving the update in Δ fails or sits between the lines only at steps 10 and 1,000, so only those two passes are
evidence. At step 10 the copy sits near the pass line, as single copies did. The chain ended with a smaller weight
change than a single 1,000-step copy, and damage just inside the limit.

## What we were trying to test

A monitor watching post-training would not copy one finished fine-tune. It would copy the update at each checkpoint,
briefly, starting from its last copy. Does that keep working, or does error pile up from copy to copy?

So we stopped the emoticon fine-tune at seven checkpoints: steps 1, 3, 10, 30, 100, 300 and 1,000. The first copy
starts from the unchanged model and trains for 100 steps to imitate the step-1 model. Each later copy starts from the
previous copy and trains 100 steps against the next checkpoint. Γ and Δ stay frozen throughout, so the update must go
into the subcomponents. We call this the short chain.

Three references sit beside each copy, on the same card and the same check.
- **Doing nothing**: the checkpoint's update put into Δ. Where this already passes the check, a passing copy tells us
  nothing about the map. Only where doing nothing fails or sits between the lines does the check see the update.
- **The one-shot copy**: E41's copy of the same checkpoint, 1,000 steps from the unchanged model, at steps 10, 100 and
  1,000. At step 10 there are three, at three batch orders.
- **E41's long chain**: copies of steps 10, 100 and 1,000 in turn, with 1,000 steps per link. It ended with a larger
  weight change and more damage than the one-shot copy, which is what this run was built to test.

The adversarial ratio is the damage a mask adversary finds on 256 Pile rows, over the unchanged model's. It reads 1 for
no change; it passes at or below 1.40 and fails at or above 1.61. **Damage** is the KL from the unchanged model at the
task windows' other positions. **Size** is the largest relative weight change of any matrix.

## What we expected, and why

- **Every copy at the line or better, most likely at the line.** Single copies of step 10 had passed once and sat
  between the lines twice. Medium confidence.
- **The chain no worse than a single copy** at steps 10, 100 and 1,000. E41's long chain had held. Medium confidence.
- **No pile-up of size or damage at the end**, though we leaned that damage would pile up. Early copies do their
  teacher's full damage, three times a finished copy's. The last copy has only 100 steps to shed it.

## Does every copy pass where doing nothing fails?

![adversarial ratio of each link against the checkpoint it copies](../experiments/teacher_training/figures/e41_amend2_faithful.png)

**Reading it.** Every copy passes, but only two of the seven passes are evidence. The green line is the short chain;
every point sits below the lower dotted pass line. The grey line is doing nothing. It passes at five checkpoints, the
pale columns, so there the check cannot tell any copy from leaving the update in Δ.

The two that count are steps 10 and 1,000. At step 1,000 the copy reads 0.91, where doing nothing sits between the lines
at 1.50: a clear pass. At step 10 doing nothing fails at 2.52, and the copy reads 1.32. That is 5% under the pass line,
inside the unchanged model's own spread across adversary starts. The three single copies of step 10, the blue diamonds,
read 1.28 to 1.55, and 1.32 falls among them.

The caveat: this is one batch order and one adversary start. The user chose to report a copy this close to the line as
at the line. So the short chain sits where single copies sit at step 10; it does not fix the early shortfall.

## Does every copy keep the behaviour?

![P(target) and gap share of each link against the checkpoint it copies](../experiments/teacher_training/figures/e41_amend2_behaviour.png)

**Reading it.** Yes, at every checkpoint. On the left, every green point holds the probability of the target token at
0.91 or more, against 0.004 for the unchanged model. From step 3 on, every copy holds 0.995 or more. The lowest is
step 1, where the teacher itself holds only 0.93. On the right, the gap share is the KL from the teacher to the copy,
as a share of the KL from the teacher to the unchanged model. Every copy sits below 0.006, far under the 0.05 line. It
creeps up along the chain and ends level with the single copy.

## Does the chain pile up size or damage?

![size and damage of each link, beside the fine-tune's own, the one-shot copies and E41's long chain](../experiments/teacher_training/figures/e41_amend2_pileup.png)

**Reading it.** No, narrowly on damage, and this reverses our lean. At step 1,000 the short chain's size is 0.83 times
the single copy's and its damage 1.20 times it. The filed limit, the dotted line, is 1.25 times. Damage sits 4% under
it, about as much as damage moves between batch orders. E41's long chain, the hollow triangles, ended at 1.36 and 1.49 times.

The green line shows how. Size grows at every copy but stays below the fine-tune's own, the grey line, from step 10 on.
Damage falls at every copy after step 3, from 0.10 to 0.022. The chain sheds the early copies' damage a little at each
step, not all in the last copy. Why the long chain did not is untested: it differs in steps per link, in total steps,
3,000 against 700, and in reusing the same batches.

## What we learned

Teacher training applied every 100 steps along this fine-tune kept every update's behaviour with Δ untouched. It
passed the adversarial check at both checkpoints where doing nothing does not. It passed clearly at step 1,000, and
sat near the line at step 10, where single copies already sat. The chain ended no worse than a single copy on every
filed comparison. It did not carry the early copies' damage forward. `GOALS.md` asks whether a map applied along training accumulates
error. In this first run at a monitor's rhythm, over seven copies on this model, it did not.

## How compelling is the evidence

- **Two copies carry the result on the check.** At the other five checkpoints doing nothing passes too.
- **One draw of everything**: one fine-tune, one task, one batch order per copy, one adversary start. A pass is a lower
  bound, and the step-10 copy could land on either side of the line at another order.
- **What each number cannot see**: the check sees only 256 Pile rows; the target probability sees only the trigger;
  the gap share and damage see only the task windows; size sees only the most-changed matrix.
- **Fewer training steps**: the chain had 700 steps at the end against the single copy's 1,000. How much of its smaller
  size is the chain, and how much is fewer steps, is untested.
- **Clean references**: this job's unchanged model, teachers and doing-nothing reading equal E41's to every digit.

## Surprises

- **The pile-up lean was reversed.** We expected damage to pile up past the limit. It ended at 1.20 times the single
  copy's, inside the limit, because the chain shed damage along the way.
- Every copy passed where we leaned "at the line", but the step-10 copy is at the line in substance.
- The step-1 copy is twice the size of its one-step update, and reads 1.17 on the check against 1.05 for doing nothing.
  Both pass.
- The step-3 checkpoint does the most damage of any, nearly twice step 10's, yet doing nothing passes the check there.

## What next

The plan's branch for "every copy passes and the chain is no worse" fires. The "continuous" cell of the technique
comparison turns yes on this model. A card asks whether to carry this rhythm to Qwen. The cell should say it rests
on two seen copies, one at the line. A single copy trained 700 steps would say whether the smaller size is the chain's.
No run is proposed here.

## Scorecard (Amendment 2)

Filed in `experiments/teacher_training/results/predictions_e41_amend2.json` before the run, refiled once before it
started, unedited since.

| id | claim | rule | read | verdict |
|---|---|---|---|---|
| C1, G1, PC, C2 | unchanged model, teachers, doing nothing at step 10, warm start reproduce | as filed | equal to E41's to the digit; warm start KL 0.0006 | pass |
| P1 | every copy keeps the behaviour and passes | worst copy's verdict | all seven pass; seen at steps 10 and 1,000 only; step 10 near the line | **pass** (lean at the line) |
| P2 | the chain is no worse than a single copy | at steps 10, 100, 1,000: verdict, ratio ≤ 1.25 ×, P ≥ −0.01, gap share ≤ +0.005 | ratio 0.86 / 0.97 / 0.95 × | **holds**, as leaned |
| P3 | no pile-up | step-1,000 size and damage ≤ 1.25 × the single copy's | 0.83 ×, 1.20 × | **holds** (lean fails on damage: reversed) |
| — | Δ untouched | Δ's share ≤ 1e-6 | ≤ 1.1e-8 | pass |

## Not run (Amendment 2)

- Other batch orders or adversary starts for any copy, by the user's choice on early updates.
- A single copy trained 700 steps; carrying Adam's state across copies; shorter links; resets.
- Chains on the " which" task, from Pile text, or of SGD or LoRA fine-tunes; a second fine-tune seed.
- No states were saved; the chain regenerates in about 8 minutes.

RunPod NVIDIA H100 80GB HBM3, 2026-10-28, about 17.5 minutes of job and 32 of pod, about $2.10. Results
`experiments/teacher_training/results/teacher_training_e41_amend2.json`; log
`experiments/teacher_training/results/large/run_e41_amend2.log`; figures from
`experiments/teacher_training/plot_e41_amend2.py`.